In [ ]:
import pandas as pd
import glob, os

In [ ]:
path = r"C:\Users\tlles\Documents\DA15\Capstone\baseball_payroll_wins_capstone\data"
files = glob.glob(os.path.join(path, "payroll_*.csv"))
for f in files:
    print(" -", os.path.basename(f))


In [18]:
dfs = []

for f in files:
    df = pd.read_csv(f)
    df.columns = [c.strip().lower() for c in df.columns]
    if df["payroll"].dtype == object:
        df["payroll"] = (
            df["payroll"]
            .astype(str)
            .str.replace(r"[\$,]", "", regex=True)
            .str.replace("M", "000000", regex=False)
            .str.replace("B", "000000000", regex=False)
            .astype(float))
    dfs.append(df)


In [19]:
combined = pd.concat(dfs, ignore_index=True)

In [20]:
combined

,team,year,payroll
0,Arizona Diamondbacks,2000,81027833.0
1,Arizona Diamondbacks,2001,81206513.0
2,Arizona Diamondbacks,2002,102820000.0
3,Arizona Diamondbacks,2003,80640333.0
4,Arizona Diamondbacks,2004,70204984.0
...,...,...,...
790,Pittsburgh Pirates,2025,75000000.0
791,Tampa Bay Rays,2025,73000000.0
792,Chicago White Sox,2025,59000000.0
793,Athletics,2025,55000000.0


In [21]:
combined = (
    combined.dropna(subset=["team", "payroll", "year"])
             .sort_values(["team", "year"])
             .reset_index(drop=True)
)

In [23]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 795 entries, 0 to 794
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   team     795 non-null    object 
 1   year     795 non-null    int64  
 2   payroll  795 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 18.8+ KB


In [24]:
combined.groupby("year").size().sort_index()

year
2000    31
2001    31
2002    31
2003    31
2004    31
2005    31
2006    31
2007    31
2008    31
2009    31
2010    31
2011    31
2012    31
2013    31
2014    31
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
2025    30
dtype: int64

In [25]:
combined[combined["team"].str.contains("Average|Total", case=False, na=False)]

,team,year,payroll
392,MLB Average,2000,55664957.0
393,MLB Average,2001,64460011.0
394,MLB Average,2002,67445550.0
395,MLB Average,2003,71028782.0
396,MLB Average,2004,68547511.0
397,MLB Average,2005,72748220.0
398,MLB Average,2006,77556890.0
399,MLB Average,2007,82633066.0
400,MLB Average,2008,89547782.0
401,MLB Average,2009,88349620.0


In [26]:
combined = combined[~combined["team"].str.contains("Average|Total", case=False, na=False)]

In [27]:
combined.groupby("year").size()

year
2000    30
2001    30
2002    30
2003    30
2004    30
2005    30
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
2012    30
2013    30
2014    30
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
2025    30
dtype: int64

In [28]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 780 entries, 0 to 794
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   team     780 non-null    object 
 1   year     780 non-null    int64  
 2   payroll  780 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 24.4+ KB


In [29]:
combined.to_csv("mlb_payroll_clean_2000_2025.csv", index=False)

In [30]:
combined["team"].nunique()

95

In [31]:
teams = sorted(combined["team"].unique())
for t in teams:
    print(t)

 Arizona Diamondbacks
 Atlanta Braves
 Baltimore Orioles
 Boston Red Sox
 Chicago Cubs
 Chicago White Sox
 Cincinnati Reds
 Cleveland Indians
 Colorado Rockies
 Detroit Tigers
 Houston Astros
 Kansas City Royals
 Los Angeles Angels
 Los Angeles Dodgers
 Miami Marlins
 Milwaukee Brewers
 Minnesota Twins
 New York Mets
 New York Yankees
 Oakland A's
 Philadelphia Phillies
 Pittsburgh Pirates
 San Diego Padres
 San Francisco Giants
 Seattle Mariners
 St. Louis Cardinals
 Tampa Bay Rays
 Texas Rangers
 Toronto Blue Jays
 Washington Nationals
Angels
Arizona Diamondbacks
Astros
Athletics
Atlanta Braves
Baltimore Orioles
Blue Jays
Blue Jays 
Boston Red Sox
Braves
Brewers
Cardinals
Chicago Cubs
Chicago White Sox
Cincinnati Reds
Cleveland Guardians
Cleveland Indians
Colorado Rockies
Cubs
Detroit Tigers
Diamondbacks
Dodgers
Florida Marlins
Giants
Guardians
Houston Astros
Indians
Kansas City Royals
Los Angeles Angels
Los Angeles Angels of Anaheim
Los Angeles Dodgers
Mariners
Marlins
Mets
Miami Ma

In [32]:
TEAM_NAME_FIXES = {
    # Angels
    "Anaheim Angels": "Los Angeles Angels",
    "Los Angeles Angels of Anaheim": "Los Angeles Angels",

    # Marlins
    "Florida Marlins": "Miami Marlins",

    # Cleveland rename
    "Cleveland Indians": "Cleveland Guardians",

    # Expos → Nationals
    "Montreal Expos": "Washington Nationals",

    # Misc. spacing or abbreviations
    "St. Louis Cardinals": "St Louis Cardinals",
    "Chicago White Sox": "Chicago White Sox",
    "Chicago Cubs": "Chicago Cubs"
}


In [34]:
combined.loc[:, "team"] = combined["team"].replace(TEAM_NAME_FIXES)

In [35]:
print(sorted(combined["team"].unique()))

[' Arizona Diamondbacks', ' Atlanta Braves', ' Baltimore Orioles', ' Boston Red Sox', ' Chicago Cubs', ' Chicago White Sox', ' Cincinnati Reds', ' Cleveland Indians', ' Colorado Rockies', ' Detroit Tigers', ' Houston Astros', ' Kansas City Royals', ' Los Angeles Angels', ' Los Angeles Dodgers', ' Miami Marlins', ' Milwaukee Brewers', ' Minnesota Twins', ' New York Mets', ' New York Yankees', " Oakland A's", ' Philadelphia Phillies', ' Pittsburgh Pirates', ' San Diego Padres', ' San Francisco Giants', ' Seattle Mariners', ' St. Louis Cardinals', ' Tampa Bay Rays', ' Texas Rangers', ' Toronto Blue Jays', ' Washington Nationals', 'Angels', 'Arizona Diamondbacks', 'Astros', 'Athletics', 'Atlanta Braves', 'Baltimore Orioles', 'Blue Jays', 'Blue Jays ', 'Boston Red Sox', 'Braves', 'Brewers', 'Cardinals', 'Chicago Cubs', 'Chicago White Sox', 'Cincinnati Reds', 'Cleveland Guardians', 'Colorado Rockies', 'Cubs', 'Detroit Tigers', 'Diamondbacks', 'Dodgers', 'Giants', 'Guardians', 'Houston Astros

In [37]:
combined.loc[:,'team'] = combined['team'].str.lstrip()

In [38]:
print(sorted(combined["team"].unique()))

['Angels', 'Arizona Diamondbacks', 'Astros', 'Athletics', 'Atlanta Braves', 'Baltimore Orioles', 'Blue Jays', 'Blue Jays ', 'Boston Red Sox', 'Braves', 'Brewers', 'Cardinals', 'Chicago Cubs', 'Chicago White Sox', 'Cincinnati Reds', 'Cleveland Guardians', 'Cleveland Indians', 'Colorado Rockies', 'Cubs', 'Detroit Tigers', 'Diamondbacks', 'Dodgers', 'Giants', 'Guardians', 'Houston Astros', 'Indians', 'Kansas City Royals', 'Los Angeles Angels', 'Los Angeles Dodgers', 'Mariners', 'Marlins', 'Mets', 'Miami Marlins', 'Milwaukee Brewers', 'Minnesota Twins', 'Nationals', 'New York Mets', 'New York Yankees', "Oakland A's", 'Oakland Athletics', 'Orioles', 'Padres', 'Philadelphia Phillies', 'Phillies', 'Pirates', 'Pittsburgh Pirates', 'Rangers', 'Rays', 'Red Sox', 'Reds', 'Rockies', 'Royals', 'San Diego Padres', 'San Francisco Giants', 'Seattle Mariners', 'St Louis Cardinals', 'St. Louis Cardinals', 'Tampa Bay Rays', 'Texas Rangers', 'Tigers', 'Toronto Blue Jays', 'Twins', 'Washington Nationals', 

In [39]:
teams = sorted(combined["team"].unique())
for t in teams:
    print(t)

Angels
Arizona Diamondbacks
Astros
Athletics
Atlanta Braves
Baltimore Orioles
Blue Jays
Blue Jays 
Boston Red Sox
Braves
Brewers
Cardinals
Chicago Cubs
Chicago White Sox
Cincinnati Reds
Cleveland Guardians
Cleveland Indians
Colorado Rockies
Cubs
Detroit Tigers
Diamondbacks
Dodgers
Giants
Guardians
Houston Astros
Indians
Kansas City Royals
Los Angeles Angels
Los Angeles Dodgers
Mariners
Marlins
Mets
Miami Marlins
Milwaukee Brewers
Minnesota Twins
Nationals
New York Mets
New York Yankees
Oakland A's
Oakland Athletics
Orioles
Padres
Philadelphia Phillies
Phillies
Pirates
Pittsburgh Pirates
Rangers
Rays
Red Sox
Reds
Rockies
Royals
San Diego Padres
San Francisco Giants
Seattle Mariners
St Louis Cardinals
St. Louis Cardinals
Tampa Bay Rays
Texas Rangers
Tigers
Toronto Blue Jays
Twins
Washington Nationals
White Sox
Yankees


In [40]:
combined = combined.copy()

In [41]:
combined["team"] = combined["team"].astype(str).str.strip()

In [42]:
combined["team"] = (combined["team"]
    .str.replace(r"^St Louis Cardinals$", "St. Louis Cardinals", regex=True)
    .str.replace(r"^Blue Jays$", "Toronto Blue Jays", regex=True)
    .str.replace(r"^Blue Jays\s+$", "Toronto Blue Jays", regex=True)
    .str.replace(r"^Oakland A['’]s$", "Oakland Athletics", regex=True)
)

In [43]:
MAP = {
    "Arizona Diamondbacks": "Arizona Diamondbacks",
    " Atlanta Braves": "Atlanta Braves",
    " Arizona Diamondbacks": "Arizona Diamondbacks",
    " Baltimore Orioles": "Baltimore Orioles",
    " Boston Red Sox": "Boston Red Sox",
    " Chicago Cubs": "Chicago Cubs",
    " Chicago White Sox": "Chicago White Sox",
    " Cincinnati Reds": "Cincinnati Reds",
    " Cleveland Indians": "Cleveland Guardians",
    " Cleveland Guardians": "Cleveland Guardians",
    " Colorado Rockies": "Colorado Rockies",
    " Detroit Tigers": "Detroit Tigers",
    " Houston Astros": "Houston Astros",
    " Kansas City Royals": "Kansas City Royals",
    " Los Angeles Angels": "Los Angeles Angels",
    " Los Angeles Dodgers": "Los Angeles Dodgers",
    " Miami Marlins": "Miami Marlins",
    " Milwaukee Brewers": "Milwaukee Brewers",
    " Minnesota Twins": "Minnesota Twins",
    " New York Mets": "New York Mets",
    " New York Yankees": "New York Yankees",
    " Philadelphia Phillies": "Philadelphia Phillies",
    " Pittsburgh Pirates": "Pittsburgh Pirates",
    " San Diego Padres": "San Diego Padres",
    " San Francisco Giants": "San Francisco Giants",
    " Seattle Mariners": "Seattle Mariners",
    " St. Louis Cardinals": "St. Louis Cardinals",
    " Tampa Bay Rays": "Tampa Bay Rays",
    " Texas Rangers": "Texas Rangers",
    " Toronto Blue Jays": "Toronto Blue Jays",
    " Washington Nationals": "Washington Nationals",
    "Angels": "Los Angeles Angels",
    "Astros": "Houston Astros",
    "Athletics": "Oakland Athletics",
    "Braves": "Atlanta Braves",
    "Brewers": "Milwaukee Brewers",
    "Cardinals": "St. Louis Cardinals",
    "Cubs": "Chicago Cubs",
    "Diamondbacks": "Arizona Diamondbacks",
    "Dodgers": "Los Angeles Dodgers",
    "Giants": "San Francisco Giants",
    "Guardians": "Cleveland Guardians",
    "Indians": "Cleveland Guardians",
    "Mariners": "Seattle Mariners",
    "Marlins": "Miami Marlins",
    "Mets": "New York Mets",
    "Nationals": "Washington Nationals",
    "Orioles": "Baltimore Orioles",
    "Padres": "San Diego Padres",
    "Phillies": "Philadelphia Phillies",
    "Pirates": "Pittsburgh Pirates",
    "Rangers": "Texas Rangers",
    "Rays": "Tampa Bay Rays",
    "Red Sox": "Boston Red Sox",
    "Reds": "Cincinnati Reds",
    "Rockies": "Colorado Rockies",
    "Royals": "Kansas City Royals",
    "Tigers": "Detroit Tigers",
    "Twins": "Minnesota Twins",
    "White Sox": "Chicago White Sox",
    "Yankees": "New York Yankees",
}

In [44]:
combined["team"] = combined ["team"].replace(MAP)

In [46]:
print("Unique team names:", len(sorted(set(combined["team"]))))

Unique team names: 31


In [47]:
for t in sorted(set(combined["team"])):
    print(t)

Arizona Diamondbacks
Atlanta Braves
Baltimore Orioles
Boston Red Sox
Chicago Cubs
Chicago White Sox
Cincinnati Reds
Cleveland Guardians
Cleveland Indians
Colorado Rockies
Detroit Tigers
Houston Astros
Kansas City Royals
Los Angeles Angels
Los Angeles Dodgers
Miami Marlins
Milwaukee Brewers
Minnesota Twins
New York Mets
New York Yankees
Oakland Athletics
Philadelphia Phillies
Pittsburgh Pirates
San Diego Padres
San Francisco Giants
Seattle Mariners
St. Louis Cardinals
Tampa Bay Rays
Texas Rangers
Toronto Blue Jays
Washington Nationals


In [51]:
combined["team"] = combined["team"].replace({"Cleveland Indians":"Cleveland Guardians"})

In [53]:
print("Unique team names:", len(sorted(set(combined["team"]))))

Unique team names: 30


In [58]:
combined.shape

(780, 3)

In [60]:
combined.to_csv("mlb_payroll_clean_2000_2025.csv",index=False)

In [61]:
combined = combined.sort_values(["year","team"]).reset_index(drop=True)

In [62]:
combined.to_csv("mlb_payroll_clean_2000_2025.csv",index=False)